In [26]:
from pyrosm import get_data, OSM
import geopandas as gpd

bbox = [-87.65, 41.87, -87.60, 41.90]  # west, south, east, north (LIST, not tuple)
pbf = "./data/chicago.osm.pbf"

osm = OSM(pbf, bounding_box=bbox)

In [3]:
buildings = osm.get_buildings()

In [ ]:
drive_net = osm.get_network(network_type="driving")

In [5]:
custom_filter = {
    "landuse": ["park", "recreation_ground"],
    "natural": ["wood", "forest", "grassland"]
}

parks_landuse = osm.get_landuse(custom_filter=custom_filter)


In [7]:
# osmium for defined bbox
# get whatever data you want with tags geometry

In [8]:
buildings.to_file("./data/buildings.geojson", driver="GeoJSON")
drive_net.to_file("./data/roads.geojson", driver="GeoJSON")
parks_landuse.to_file("./data/parks.geojson", driver="GeoJSON")

In [9]:
buildings.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [15]:
buildings[['id', 'timestamp', 'version', 'tags',
       'osm_type', 'geometry', 'changeset'
]]

,id,timestamp,version,tags,osm_type,geometry,changeset
0,23807294,1758646090,40,"{""addr:state"":""IL"",""addr:street:name"":""Clinton...",way,"POLYGON ((-87.63995 41.88379, -87.63993 41.883...",NaN
1,24800223,1717519756,14,"{""addr:state"":""IL"",""addr:street:name"":""Chicago...",way,"POLYGON ((-87.62157 41.89747, -87.62088 41.897...",NaN
2,24989834,1758228174,23,"{""addr:state"":""IL"",""addr:street:name"":""Harriso...",way,"POLYGON ((-87.63803 41.87453, -87.63809 41.874...",NaN
3,25885878,1744497182,12,"{""addr:state"":""IL"",""addr:street:name"":""Harriso...",way,"POLYGON ((-87.62582 41.87433, -87.62555 41.874...",NaN
4,28292694,1747339495,22,"{""access"":""yes"",""addr:state"":""IL"",""addr:street...",way,"POLYGON ((-87.62857 41.87583, -87.62858 41.875...",NaN
...,...,...,...,...,...,...,...
3398,69296590937,1731444838,2,"{""addr:state"":""IL"",""heating"":""yes"",""layer"":""1""...",relation,"MULTIPOLYGON (((-87.63554 41.89649, -87.63547 ...",0.0
3399,82342497746,1753393966,1,"{""addr:state"":""IL"",""building:colour"":""#4D5143""...",relation,"POLYGON ((-87.61378 41.89182, -87.61279 41.891...",0.0
3400,84267326817,1756171427,1,"{""addr:state"":""IL"",""addr:street:name"":""Huron"",...",relation,"POLYGON ((-87.62666 41.89456, -87.62667 41.894...",0.0
3401,84325926595,1756414824,2,"{""type"":""multipolygon""}",relation,"POLYGON ((-87.62177 41.87866, -87.62177 41.878...",0.0


In [24]:
buildings[buildings['id'] == 1271821350][['id', 'timestamp', 'version', 'tags',
       'osm_type', 'geometry', 'changeset'
]]

,id,timestamp,version,tags,osm_type,geometry,changeset


In [27]:
def get_buildings_by_name(region_key, bbox, name_substring):
    """
    region_key: Geofabrik key pyrosm knows (e.g., 'virginia', 'california', 'chicago')
    bbox: [west, south, east, north] in EPSG:4326
    name_substring: case-insensitive match on 'name' tag
    """
    pbf_path = get_data(region_key)                 # downloads once to ~/.pyrosm/cache
    osm = OSM(pbf_path, bounding_box=bbox)          # fast: only parses inside bbox
    b = osm.get_buildings()                         # GeoDataFrame (Multi)Polygons; holes preserved
    if b is None or b.empty:
        return gpd.GeoDataFrame(columns=["name", "geometry"], crs="EPSG:4326")
    return b[b["name"].str.contains(name_substring, case=False, na=False)]

pentagon_bbox = [-77.0609, 38.8683, -77.0534, 38.8735]
pentagon = get_buildings_by_name("virginia", pentagon_bbox, "Pentagon")

Downloaded Protobuf data 'virginia-latest.osm.pbf' (394.83 MB) to:
'C:\Users\qshah\AppData\Local\Temp\pyrosm\virginia-latest.osm.pbf'


In [28]:
pentagon

,addr:city,addr:country,addr:housenumber,addr:postcode,addr:street,name,visible,building,amenity,building:levels,...,version,geometry,tags,osm_type,operator,phone,website,height,office,changeset
13,NaN,NaN,NaN,NaN,NaN,The Pentagon,NaN,government,NaN,4,...,65,"POLYGON ((-77.05844 38.87004, -77.0583 38.8706...","{""access"":""private"",""building:colour"":""#F5DEB3...",relation,United States Department of Defense,+1 703-697-1776,https://pentagontours.osd.mil/,23,government,0.0
